# Experiment-8


## To implement and evaluate abstractive text summarization using two multilingual Transformer-based sequence-to-sequence models:
1.	mT5 (Multilingual Text-to-Text Transfer Transformer)
2.	mBART (Multilingual Bidirectional and Auto-Regressive Transformers)

Use on any one Language (e.g, Hindi/Nepali/Bengali/Telugu etc.) of the XL-Sum (Cross-Lingual Summarization) dataset to compare the performance of both models using automatic evaluation metrics (ROUGE-1, ROUGE-2 and ROUGE-L and mBERT scores).


In [ ]:
# Install required libraries
!pip install -q datasets transformers[torch] sentencepiece evaluate rouge-score bert-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


```markdown
## 1. Load the Dataset
We will use the **Bengali** subset of the `csebuetnlp/xlsum` dataset.
```

In [ ]:
import os
import glob
from google.colab import drive

# Mount Google Drive to access the dataset
if not os.path.exists("/content/drive"):
    print("Mounting Google Drive...")
    drive.mount("/content/drive")

archive_path = "/content/drive/MyDrive/CNLP-Lab/Experiment-8/bengali_XLSum_v2.0.tar.bz2"

if os.path.exists(archive_path):
    print(f"Found archive: {archive_path}")
    print("Extracting dataset files...")
    # Extract to a dedicated local directory
    os.makedirs("/content/bengali_dataset", exist_ok=True)
    !tar -xf "{archive_path}" -C /content/bengali_dataset

    # List extracted files to verify
    extracted_files = glob.glob("/content/bengali_dataset/**/*.jsonl", recursive=True)
    print("Extracted JSONL files:", extracted_files)
else:
    print(f"Error: The file could not be found at {archive_path}. Please check your Google Drive paths.")

Found archive: /content/drive/MyDrive/CNLP-Lab/Experiment-8/bengali_XLSum_v2.0.tar.bz2
Extracting dataset files...
Extracted JSONL files: ['/content/bengali_dataset/bengali_test.jsonl', '/content/bengali_dataset/bengali_val.jsonl', '/content/bengali_dataset/bengali_train.jsonl']


In [ ]:
# Load the locally extracted jsonl files into the Hugging Face dataset format
train_files = glob.glob("/content/bengali_dataset/**/*_train.jsonl", recursive=True)
test_files = glob.glob("/content/bengali_dataset/**/*_test.jsonl", recursive=True)
val_files = glob.glob("/content/bengali_dataset/**/*_val.jsonl", recursive=True)

if train_files and test_files and val_files:
    dataset = load_dataset("json", data_files={
        "train": train_files[0],
        "test": test_files[0],
        "validation": val_files[0]
    })
    print("Dataset successfully loaded from your Google Drive archive!")
    print(dataset)
else:
    print("Could not automatically locate the extracted jsonl files. Please run the extraction cell again or verify the archive's internal structure.")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Dataset successfully loaded from your Google Drive archive!
DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 8102
    })
    test: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 1012
    })
    validation: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 1012
    })
})


```markdown
## 2. Model Setup
We load pretrained seq2seq models. We will target existing Bengali fine-tuned models to demonstrate robust performance directly, or generic checkpoints if needed:
- **mT5**: `csebuetnlp/mT5_multilingual_XLSum` (or a base version)
- **mBART**: `facebook/mbart-large-50` (or a fine-tuned version on multilingual summarization)
```

In [ ]:
# Define model checkpoints
mt5_checkpoint = "csebuetnlp/mT5_multilingual_XLSum"  # Already fine-tuned on XL-Sum
mbart_checkpoint = "facebook/mbart-large-50-many-to-many-mmt"

# Load mT5 tokenizer and model
print("Loading mT5 Tokenizer and Model...")
mt5_tokenizer = AutoTokenizer.from_pretrained(mt5_checkpoint, use_fast=False)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(mt5_checkpoint).to(device)

# Load mBART tokenizer and model
print("Loading mBART Tokenizer and Model...")
mbart_tokenizer = AutoTokenizer.from_pretrained(mbart_checkpoint)
mbart_model = AutoModelForSeq2SeqLM.from_pretrained(mbart_checkpoint).to(device)

Loading mT5 Tokenizer and Model...


config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

... (12 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.44GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

```markdown
## 3. Evaluation Setup & Functions
We will sample a subset of the test set for evaluation to demonstrate the pipeline speed. We then define a generation function and evaluate the results.
```

In [ ]:
# Select a small sample for demonstration to avoid long execution times
eval_subset = dataset["test"].select(range(10))

# Load metrics
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

def generate_summary(model, tokenizer, text, model_type="mt5"):
    if model_type == "mbart":
        # Set Bengali source language token for mBART-50
        tokenizer.src_lang = "bn_IN"

    inputs = tokenizer(text, max_length=512, truncation=True, return_tensors="pt").to(device)

    # Generate summary tokens
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=84,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [ ]:
import tqdm

references = eval_subset["summary"]
texts = eval_subset["text"]

mt5_predictions = []
mbart_predictions = []

print("Generating summaries with mT5...")
for t in tqdm.tqdm(texts):
    mt5_predictions.append(generate_summary(mt5_model, mt5_tokenizer, t, model_type="mt5"))

print("\nGenerating summaries with mBART...")
for t in tqdm.tqdm(texts):
    mbart_predictions.append(generate_summary(mbart_model, mbart_tokenizer, t, model_type="mbart"))

Generating summaries with mT5...


100%|██████████| 10/10 [00:24<00:00,  2.44s/it]



Generating summaries with mBART...


100%|██████████| 10/10 [00:17<00:00,  1.77s/it]


```markdown
## 4. Compare Performance
Now, we compute and compare the ROUGE score and Multilingual BERT score for both models.
```

In [ ]:
def compute_metrics(predictions, references):
    # Compute ROUGE
    rouge_results = rouge_metric.compute(predictions=predictions, references=references)

    # Compute Multilingual BERTScore (using bert-base-multilingual-cased)
    bert_results = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        model_type="bert-base-multilingual-cased"
    )

    # Average of F1 scores from BERTScore
    avg_bert_f1 = np.mean(bert_results["f1"])

    results = {
        "ROUGE-1": rouge_results["rouge1"],
        "ROUGE-2": rouge_results["rouge2"],
        "ROUGE-L": rouge_results["rougeL"],
        "mBERT F1": avg_bert_f1
    }
    return results

print("Calculating mT5 metrics...")
mt5_scores = compute_metrics(mt5_predictions, references)

print("Calculating mBART metrics...")
mbart_scores = compute_metrics(mbart_predictions, references)

# Combine and display as a DataFrame
comparison_df = pd.DataFrame([mt5_scores, mbart_scores], index=["mT5", "mBART"])
display(comparison_df)

Calculating mT5 metrics...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Calculating mBART metrics...


,ROUGE-1,ROUGE-2,ROUGE-L,mBERT F1
mT5,0.0,0.0,0.0,0.783557
mBART,0.0,0.0,0.0,0.597714
